# Incremental Silver Processing — Data Quality and Delta MERGE

**Objective:** Process newly ingested Bronze transactions, apply data quality rules, identify new valid records, and incrementally update the Silver Delta table using PySpark and Delta MERGE.

### Why?

**For our project:** Incorporates new financial transactions while preventing invalid records and duplicate transaction IDs from affecting downstream KPIs.

**For Data Engineering:** Demonstrates incremental data processing, Delta Time Travel, data quality enforcement, anti-joins, MERGE operations, and idempotency validation.

#### 01 — Identify Incremental Bronze Records

Compare Bronze Delta versions 2 and 3 using Time Travel and `exceptAll()` to identify the eight records introduced by Batch 02. This isolates the incremental input without reprocessing the entire historical dataset.

In [0]:
# Read the Bronze snapshot before Batch 02 ingestion
df_bronze_before = spark.read.option("versionAsOf", 2).table("workspace.default.fintech_transactions_bronze")
display(df_bronze_before)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX001,C001,2026-10-18 09:00:00,PIX_IN,250.00,COMPLETED
TX002,C001,2026-10-18 09:30:00,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18 10:00:00,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18 11:00:00,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18 12:00:00,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18 13:00:00,PIX_OUT,120.00,FAILED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX008,C006,2026-10-18 14:00:00,PIX_OUT,35.00,COMPLETED
null,C007,2026-10-18 14:15:00,CARD_PURCHASE,25.00,COMPLETED


In [0]:
assert df_bronze_before.count() == 12, "Unexpected Bronze Before Dataframe row count"

In [0]:
# Read the Bronze snapshot after Batch 02 ingestion
df_bronze_after = spark.read.option("versionAsOf", 3).table("workspace.default.fintech_transactions_bronze")
display(df_bronze_after)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX001,C001,2026-10-18 09:00:00,PIX_IN,250.00,COMPLETED
TX002,C001,2026-10-18 09:30:00,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18 10:00:00,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18 11:00:00,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18 12:00:00,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18 13:00:00,PIX_OUT,120.00,FAILED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX008,C006,2026-10-18 14:00:00,PIX_OUT,35.00,COMPLETED
null,C007,2026-10-18 14:15:00,CARD_PURCHASE,25.00,COMPLETED


In [0]:
assert df_bronze_after.count() == 20, "Unexpected Bronze After Dataframe row count"

In [0]:
# Identify newly ingested rows while preserving duplicate occurrences
df_bronze_increment = df_bronze_after.exceptAll(df_bronze_before)
display(df_bronze_increment)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX003,C002,2026-10-18 10:00:00,CARD_PURCHASE,90.50,COMPLETED
TX012,C002,2026-10-19 10:00:00,CARD_PURCHASE,45.25,COMPLETED
TX014,C004,2026-10-19 12:00:00,PIX_OUT,30.00,PENDING
TX016,C006,2026-10-19 14:00:00,TRANSFER,35.00,COMPLETED
TX011,C001,2026-10-19 09:00:00,PIX_IN,150.00,COMPLETED
TX013,C003,2026-10-19 11:00:00,BILL_PAYMENT,60.00,COMPLETED
TX015,C005,2026-10-19 13:00:00,PIX_IN,900.00,FAILED
TX017,C007,2026-10-19 15:00:00,PIX_IN,300.00,COMPLETED


In [0]:
assert df_bronze_increment.count() == 8, "Unexpected Bronze Increment table row count"

#### 02 — Standardize and Validate Transactions

Convert incoming transaction amounts and timestamps to the required Silver data types. Apply the six existing Data Quality Rules (DQRs) and separate valid and invalid records.

Preserve record multiplicities during validation to ensure that all incoming transactions are accounted for.

In [0]:
import pyspark.sql.functions as sf

# Convert monetary values to fixed-precision decimals
df_increment_typed = df_bronze_increment.withColumn("amount", sf.col("amount").cast("decimal(12,2)"))    
display(df_increment_typed)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX003,C002,2026-10-18 10:00:00,CARD_PURCHASE,90.50,COMPLETED
TX012,C002,2026-10-19 10:00:00,CARD_PURCHASE,45.25,COMPLETED
TX014,C004,2026-10-19 12:00:00,PIX_OUT,30.00,PENDING
TX016,C006,2026-10-19 14:00:00,TRANSFER,35.00,COMPLETED
TX011,C001,2026-10-19 09:00:00,PIX_IN,150.00,COMPLETED
TX013,C003,2026-10-19 11:00:00,BILL_PAYMENT,60.00,COMPLETED
TX015,C005,2026-10-19 13:00:00,PIX_IN,900.00,FAILED
TX017,C007,2026-10-19 15:00:00,PIX_IN,300.00,COMPLETED


In [0]:
import pyspark.sql.functions as sf

# Parse transaction timestamps for reliable date-based processing
df_increment_typed = df_increment_typed.withColumn("transaction_ts", sf.to_timestamp(sf.col("transaction_ts"), "yyyy-MM-dd HH:mm:ss"))    
display(df_increment_typed)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX012,C002,2026-10-19T10:00:00.000Z,CARD_PURCHASE,45.25,COMPLETED
TX014,C004,2026-10-19T12:00:00.000Z,PIX_OUT,30.00,PENDING
TX016,C006,2026-10-19T14:00:00.000Z,TRANSFER,35.00,COMPLETED
TX011,C001,2026-10-19T09:00:00.000Z,PIX_IN,150.00,COMPLETED
TX013,C003,2026-10-19T11:00:00.000Z,BILL_PAYMENT,60.00,COMPLETED
TX015,C005,2026-10-19T13:00:00.000Z,PIX_IN,900.00,FAILED
TX017,C007,2026-10-19T15:00:00.000Z,PIX_IN,300.00,COMPLETED


#### DQR-001/002:

Reject missing transaction or customer IDs

In [0]:
import pyspark.sql.functions as sf

df_invalid_ids = df_increment_typed.filter( sf.col("transaction_id").isNull() | sf.col("customer_id").isNull() )
display(df_invalid_ids)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status


In [0]:
assert df_invalid_ids.count() == 0, "Unexpected invalid identifier count"

#### DQR-003: 

Reject null, zero, or negative amounts

In [0]:
import pyspark.sql.functions as sf

df_invalid_amounts = df_increment_typed.filter( sf.col("amount").isNull() | (sf.col("amount") <= 0) )
display(df_invalid_amounts)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status


In [0]:
assert df_invalid_amounts.count() == 0,  "Unexpected invalid amount count"

#### DQR-004: 

Reject invalid transaction timestamps

In [0]:
import pyspark.sql.functions as sf

df_invalid_timestamps = df_increment_typed.filter( sf.col("transaction_ts").isNull())
display(df_invalid_timestamps)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status


In [0]:
assert df_invalid_timestamps.count() == 0, "Unexpected invalid transaction_ts count"

#### DQR-005:

Reject unsupported transaction types

In [0]:
import pyspark.sql.functions as sf

df_invalid_transaction_types = df_increment_typed.filter( sf.col("transaction_type").isNull() | ~ ( sf.col("transaction_type").isin(["PIX_IN", "PIX_OUT", "CARD_PURCHASE", "BILL_PAYMENT"]) ) )
display(df_invalid_transaction_types)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX016,C006,2026-10-19T14:00:00.000Z,TRANSFER,35.00,COMPLETED


In [0]:
assert df_invalid_transaction_types.count() == 1, "Unexpected invalid transaction type count"

#### DQR-006:

Reject unsupported transaction statuses

In [0]:
import pyspark.sql.functions as sf

df_invalid_statuses = df_increment_typed.filter( sf.col("status").isNull() | ~ ( sf.col("status").isin(["COMPLETED", "PENDING", "FAILED"]) ) )
display(df_invalid_statuses)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status


In [0]:
assert df_invalid_statuses.count() == 0, "Unexpected invalid status type count"

In [0]:
# Separate valid and invalid rows while preserving duplicate occurrences
df_increment_typed_valid = df_increment_typed

for df_increment_typed_invalid in [
    df_invalid_ids,
    df_invalid_amounts,
    df_invalid_timestamps,
    df_invalid_transaction_types,
    df_invalid_statuses
]:
    df_increment_typed_valid = df_increment_typed_valid.exceptAll(df_increment_typed_invalid)

df_increment_typed_invalid = df_increment_typed.exceptAll(df_increment_typed_valid)

In [0]:
# Validate valid and invalid record counts and ensure all incoming records are accounted for
assert df_increment_typed_valid.count() == 7
assert df_increment_typed_invalid.count() == 1

assert (
    df_increment_typed_valid.count() + df_increment_typed_invalid.count()
    == df_increment_typed.count()
)

In [0]:
display(df_increment_typed_valid)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX013,C003,2026-10-19T11:00:00.000Z,BILL_PAYMENT,60.00,COMPLETED
TX015,C005,2026-10-19T13:00:00.000Z,PIX_IN,900.00,FAILED
TX017,C007,2026-10-19T15:00:00.000Z,PIX_IN,300.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX014,C004,2026-10-19T12:00:00.000Z,PIX_OUT,30.00,PENDING
TX011,C001,2026-10-19T09:00:00.000Z,PIX_IN,150.00,COMPLETED
TX012,C002,2026-10-19T10:00:00.000Z,CARD_PURCHASE,45.25,COMPLETED


In [0]:
display(df_increment_typed_invalid)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX016,C006,2026-10-19T14:00:00.000Z,TRANSFER,35.00,COMPLETED


#### 03 — Identify New Silver Transactions

Compare valid incoming transaction IDs with the existing Silver table using a left_anti join. This identifies six new transactions while excluding TX003, which was already processed.

This prevents existing transaction IDs from being included in the new-record dataset.

In [0]:
# Read the existing Silver transactions for comparison
df_silver = spark.table("workspace.default.fintech_transactions_silver")  
display(df_silver)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX002,C001,2026-10-18T09:30:00.000Z,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18T11:00:00.000Z,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18T12:00:00.000Z,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18T13:00:00.000Z,PIX_OUT,120.00,FAILED
TX008,C006,2026-10-18T14:00:00.000Z,PIX_OUT,35.00,COMPLETED
TX001,C001,2026-10-18T09:00:00.000Z,PIX_IN,250.00,COMPLETED


In [0]:
display(df_increment_typed_valid)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX013,C003,2026-10-19T11:00:00.000Z,BILL_PAYMENT,60.00,COMPLETED
TX015,C005,2026-10-19T13:00:00.000Z,PIX_IN,900.00,FAILED
TX017,C007,2026-10-19T15:00:00.000Z,PIX_IN,300.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX014,C004,2026-10-19T12:00:00.000Z,PIX_OUT,30.00,PENDING
TX011,C001,2026-10-19T09:00:00.000Z,PIX_IN,150.00,COMPLETED
TX012,C002,2026-10-19T10:00:00.000Z,CARD_PURCHASE,45.25,COMPLETED


In [0]:
# Identify valid incoming transactions not yet present in Silver
df_increment_new = df_increment_typed_valid.join(
    df_silver.select("transaction_id"),
    on="transaction_id",
    how="left_anti"
)

display(df_increment_new)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX013,C003,2026-10-19T11:00:00.000Z,BILL_PAYMENT,60.00,COMPLETED
TX015,C005,2026-10-19T13:00:00.000Z,PIX_IN,900.00,FAILED
TX017,C007,2026-10-19T15:00:00.000Z,PIX_IN,300.00,COMPLETED
TX014,C004,2026-10-19T12:00:00.000Z,PIX_OUT,30.00,PENDING
TX011,C001,2026-10-19T09:00:00.000Z,PIX_IN,150.00,COMPLETED
TX012,C002,2026-10-19T10:00:00.000Z,CARD_PURCHASE,45.25,COMPLETED


In [0]:
assert df_increment_new.count() == 6, \
    "Unexpected new transaction count"

#### 04 — Merge Valid Transactions into Silver

Use an insert-only Delta MERGE to add valid transactions that do not already exist in Silver, matching records by `transaction_id`.

This preserves existing transactions and prevents duplicate IDs when the same valid batch is processed again.

In [0]:
df_silver = spark.table("workspace.default.fintech_transactions_silver")  
display(df_silver)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX002,C001,2026-10-18T09:30:00.000Z,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18T11:00:00.000Z,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18T12:00:00.000Z,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18T13:00:00.000Z,PIX_OUT,120.00,FAILED
TX008,C006,2026-10-18T14:00:00.000Z,PIX_OUT,35.00,COMPLETED
TX001,C001,2026-10-18T09:00:00.000Z,PIX_IN,250.00,COMPLETED


In [0]:
display(df_increment_typed_valid)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX013,C003,2026-10-19T11:00:00.000Z,BILL_PAYMENT,60.00,COMPLETED
TX015,C005,2026-10-19T13:00:00.000Z,PIX_IN,900.00,FAILED
TX017,C007,2026-10-19T15:00:00.000Z,PIX_IN,300.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX014,C004,2026-10-19T12:00:00.000Z,PIX_OUT,30.00,PENDING
TX011,C001,2026-10-19T09:00:00.000Z,PIX_IN,150.00,COMPLETED
TX012,C002,2026-10-19T10:00:00.000Z,CARD_PURCHASE,45.25,COMPLETED


In [0]:
import pyspark.sql.functions as sf

# Verify that valid incoming transaction IDs are unique
assert df_increment_typed_valid.select("transaction_id").distinct().count() == 7, \
    "Unexpected distinct transaction_id values count"

In [0]:
display(df_increment_new)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX013,C003,2026-10-19T11:00:00.000Z,BILL_PAYMENT,60.00,COMPLETED
TX015,C005,2026-10-19T13:00:00.000Z,PIX_IN,900.00,FAILED
TX017,C007,2026-10-19T15:00:00.000Z,PIX_IN,300.00,COMPLETED
TX014,C004,2026-10-19T12:00:00.000Z,PIX_OUT,30.00,PENDING
TX011,C001,2026-10-19T09:00:00.000Z,PIX_IN,150.00,COMPLETED
TX012,C002,2026-10-19T10:00:00.000Z,CARD_PURCHASE,45.25,COMPLETED


In [0]:
display(df_increment_typed_valid)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX013,C003,2026-10-19T11:00:00.000Z,BILL_PAYMENT,60.00,COMPLETED
TX015,C005,2026-10-19T13:00:00.000Z,PIX_IN,900.00,FAILED
TX017,C007,2026-10-19T15:00:00.000Z,PIX_IN,300.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX014,C004,2026-10-19T12:00:00.000Z,PIX_OUT,30.00,PENDING
TX011,C001,2026-10-19T09:00:00.000Z,PIX_IN,150.00,COMPLETED
TX012,C002,2026-10-19T10:00:00.000Z,CARD_PURCHASE,45.25,COMPLETED


In [0]:
from delta.tables import DeltaTable

# Insert only valid transactions with unmatched transaction IDs
silver_table = DeltaTable.forName(
    spark,
    "workspace.default.fintech_transactions_silver"
)

(
    silver_table.alias("target")
    .merge(
        df_increment_typed_valid.alias("source"),
        "target.transaction_id = source.transaction_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

#### 05 — Validate Silver Persistence and Idempotency

Verify that the updated Silver table contains 14 unique transactions and preserves valid transaction statuses.

Inspect Delta transaction history to confirm that the initial MERGE inserted six records and that a repeated MERGE inserted no additional records. Validate that existing IDs are not duplicated and invalid transactions are excluded.

In [0]:
# Validate the persisted Silver row count and transaction ID uniqueness
df_silver_updated = spark.table(
    "workspace.default.fintech_transactions_silver"
)

assert df_silver_updated.count() == 14
assert df_silver_updated.select("transaction_id").distinct().count() == 14

display(df_silver_updated)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX002,C001,2026-10-18T09:30:00.000Z,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18T11:00:00.000Z,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18T12:00:00.000Z,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18T13:00:00.000Z,PIX_OUT,120.00,FAILED
TX008,C006,2026-10-18T14:00:00.000Z,PIX_OUT,35.00,COMPLETED
TX001,C001,2026-10-18T09:00:00.000Z,PIX_IN,250.00,COMPLETED
TX013,C003,2026-10-19T11:00:00.000Z,BILL_PAYMENT,60.00,COMPLETED
TX015,C005,2026-10-19T13:00:00.000Z,PIX_IN,900.00,FAILED


In [0]:
%sql
-- Inspect Delta MERGE history to verify insertion metrics and idempotency
DESCRIBE HISTORY workspace.default.fintech_transactions_silver;

In [0]:
# Verify duplicate prevention, invalid-record exclusion, and valid FAILED status preservation
from pyspark.sql import functions as sf

assert df_silver_updated.filter(
    sf.col("transaction_id") == "TX003"
).count() == 1

assert df_silver_updated.filter(
    sf.col("transaction_id") == "TX016"
).count() == 0

assert df_silver_updated.filter(
    (sf.col("transaction_id") == "TX015") &
    (sf.col("status") == "FAILED")
).count() == 1

#### Engineering Note — Incremental Processing and Idempotency

This notebook uses an insert-only Delta MERGE based on `transaction_id`. Reprocessing the same valid records does not create additional Silver transactions.

This approach supports idempotent insertion but does not update existing transactions or handle changes to previously processed records.